# 01_EDA

Перенос исходных ячеек, исторические результаты validation. Черновик: зависимости и повторный запуск ещё не проверены.

[Colab — основной источник](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%22ru_sentiment_pipeline_adapted_gzip_robust_ipynb%22.ipynb)

[Colab — нейросетевой источник](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22ru_sentiment_pipeline_adapted_gzip_robust_ipynb%22.ipynb)

Ссылки открывают загруженные исходники в Colab; исходные Drive URL требуют подтверждения.

**Задача:** трёхклассовая классификация тональности (Нейтральный / Позитивный / Негативный).

**Датасет:** `merged_sentiment_dataset.csv.gz` (скачивается по ID из Google Drive).

**Маппинг классов:** `0 = Нейтральный`, `1 = Позитивный`, `2 = Негативный`

**Часть I — EDA, предобработка, честный split, классические модели:**
- EDA: распределения классов, источники, разбиение, длины, дубликаты
- Препроцессинг (эмодзи и `!?()` сохраняются, стоп-слова не удаляются)
- Честное разбиение train/val/test
- Классика (NB × 3, LogReg, LinearSVC, NB-SVM) с подбором гиперпараметров
- Интерактивные графики Plotly

**Часть II — углублённые эксперименты:**
- Лемматизация как отдельный эксперимент
- Расширенный набор векторизаций (BoW, TF-IDF, sublinear, char, word+char)
- Подбор гиперпараметров через Optuna (TPE) для LogReg / LinearSVC / SGD
- Нейросетевой компонент: PyTorch MLP на TF-IDF
- Ансамбли: soft voting, weighted average, stacking
- Проверки на переобучение (learning curves)
- Детальный анализ ошибок и разбор спорной разметки

---
## EDA: обзор датасета

Сначала смотрим на данные (распределение классов, длины, примеры), затем — очистка.


## 0. Окружение

In [ ]:
!pip install -q gdown lightgbm optuna plotly kaleido pymorphy3

# fastText (если ещё нужен из части I)
try:
    import fasttext
except ImportError:
    !pip install -q setuptools wheel pybind11
    !pip install -q git+https://github.com/facebookresearch/fastText.git

In [ ]:
import os, re, time, random, copy, warnings
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import gdown

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.decomposition import TruncatedSVD
from sklearn.naive_bayes import MultinomialNB, ComplementNB, BernoulliNB
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, learning_curve,
)
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_colwidth", 150)
sns.set_style("whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Классы и их маппинг ----
CLASS_MAP   = {0: "Нейтральный", 1: "Позитивный", 2: "Негативный"}
CLASS_ORDER = [0, 1, 2]
CLASS_NAMES = [CLASS_MAP[i] for i in CLASS_ORDER]

# Алиасы для совместимости с кодом части I (коллега) и части II (наш)
LABELS      = CLASS_NAMES
LABELS_RU   = CLASS_NAMES
id2label    = CLASS_MAP
label2id    = {v: k for k, v in CLASS_MAP.items()}

# ---- Прочее ----
LEADERBOARD = []
RANDOM_STATE = 42

def report(name, y_true, y_pred, train_time=0.0, group="—"):
    """Считает macro-F1 и accuracy, добавляет строку в leaderboard."""
    from sklearn.metrics import f1_score, accuracy_score
    f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    LEADERBOARD.append({"группа": group, "модель": name,
                        "macro-F1": round(f1, 4), "accuracy": round(acc, 4),
                        "обучение, с": round(train_time, 1)})
    print(f"{name:55s}  macro-F1={f1:.4f}  acc={acc:.4f}  ({train_time:.1f} с)")
    return f1

# Проверка, что всё на месте
assert LABELS == ["Нейтральный", "Позитивный", "Негативный"]
assert label2id["Позитивный"] == 1
print("Константы классов:")
print(f"  CLASS_MAP   = {CLASS_MAP}")
print(f"  LABELS      = {LABELS}")
print(f"  label2id    = {label2id}")

---
## 1. Данные: `merged_sentiment_dataset.csv.gz`

Ноутбук читает как обычный CSV, так и настоящий gzip-архив. Перед чтением он проверяет сигнатуру файла, чтобы не пытаться распаковать обычный CSV только потому, что в имени есть `.gz`. Если архив повреждён, выводится понятная ошибка; при наличии рядом исходного CSV предусмотрен автоматический переход на него.

В датасете уже есть поле `split`: обучающая часть (`train`) и отложенная проверочная часть (`validation`). Это разбиение сохраняется.


In [ ]:
FILE_ID = "17KTVvvyPAZL10rCm22Je_xFqiwJXy1Me"
FILE_NAME = "merged_sentiment_dataset.csv.gz"

if not Path(FILE_NAME).exists():
    gdown.download(id=FILE_ID, output=FILE_NAME, quiet=False)

# Проверка сигнатуры gzip
with open(FILE_NAME, "rb") as f:
    magic = f.read(2)

if magic == b"\x1f\x8b":
    compression = "gzip"
    print("Файл является gzip-архивом — читаем с compression='gzip'")
else:
    compression = None
    print("Файл НЕ gzip — читаем как обычный CSV")
    # если расширение .gz, но сигнатуры нет — переименуем
    if FILE_NAME.endswith(".gz"):
        new_name = FILE_NAME[:-3]
        Path(FILE_NAME).rename(new_name)
        FILE_NAME = new_name

df = pd.read_csv(FILE_NAME, compression=compression, low_memory=False)

print(f"\nРазмер: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
df.head(3)

In [ ]:
required = {"text", "sentiment", "split"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Отсутствуют обязательные колонки: {missing}")

# Убеждаемся, что sentiment — числа 0/1/2
df["sentiment"] = pd.to_numeric(df["sentiment"], errors="coerce")
df = df.dropna(subset=["sentiment", "text", "split"]).copy()
df["sentiment"] = df["sentiment"].astype(int)

uniq = sorted(df["sentiment"].unique())
print(f"Уникальные sentiment: {uniq}")
assert set(uniq).issubset({0, 1, 2}), f"Неизвестные значения: {uniq}"

df["sentiment_name"] = df["sentiment"].map(CLASS_MAP)
assert df["sentiment_name"].isnull().sum() == 0

print("\n=== Итоговое распределение классов ===")
counts = df["sentiment_name"].value_counts().reindex(LABELS)
shares = (counts / counts.sum() * 100).round(2)
print(pd.DataFrame({"count": counts, "share, %": shares}))

print("\n=== Разбиение ===")
print(df["split"].value_counts())

if "label_source" in df.columns:
    print("\n=== label_source × split ===")
    print(pd.crosstab(df["label_source"], df["split"], margins=True))

# EDA

In [ ]:
# Для последующего EDA используем названия классов на русском.
# Поле text — единый источник текста для train и validation.
df["full_text"] = df["text"].fillna("").astype(str).str.strip()
df["sentiment_name"] = df["sentiment"].map(CLASS_MAP)

print("Пример full_text:")
print(df.loc[df["full_text"].str.len().gt(0), "full_text"].iloc[0][:500])

if "source_dataset" in df.columns:
    print("\nИсточники данных:")
    display(pd.crosstab(df["source_dataset"], df["split"], margins=True))
if "label_source" in df.columns:
    print("\nИсточники разметки:")
    display(pd.crosstab(df["label_source"], df["split"], margins=True))


### Базовая информация


In [ ]:
import re

pd.set_option('display.max_colwidth', 150)
sns.set_style('whitegrid')

print('Размер датасета:', df.shape)
print('\n=== Первые строки ===')
display(df.head())

print('\n=== Информация о колонках ===')
print(df.info())

print('\n=== Пропуски по колонкам ===')
print(df.isnull().sum())

In [ ]:
print("=== Краткая статистика ===")
print(f"Количество строк: {len(df):,}")
print(f"Пустых текстов: {df['full_text'].str.strip().eq('').sum():,}")
print("\nПропуски в основных полях:")
display(df[[c for c in ["text", "sentiment", "split", "source_dataset", "label_source", "rating", "pros", "cons"] if c in df.columns]].isna().sum().to_frame("пропуски"))

df["text_length"] = df["full_text"].str.len()
print("\nДлина текста (символы):")
display(df["text_length"].describe(percentiles=[.5, .9, .95, .99]).to_frame().T)


In [ ]:
print('=== Распределение классов ===')
print(df['sentiment'].value_counts().sort_index())
print('\nДоли (%):')
print((df['sentiment'].value_counts(normalize=True).sort_index() * 100).round(2))

# Если sentiment_name уже есть — используем его
order = ['Нейтральный', 'Позитивный', 'Негативный']
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='sentiment_name', order=order, palette='viridis')
plt.title('Распределение классов в датасете')
plt.xlabel('Тональность')
plt.ylabel('Количество отзывов')
plt.show()

print(f'\nВсего: {len(df)}')

In [ ]:
# full_text уже есть; если нет — берём text
if 'full_text' not in df.columns:
    df['full_text'] = df['text'].astype(str)

df['text_length'] = df['full_text'].astype(str).str.len()

print('=== Статистика длины ===')
print(df['text_length'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='text_length', bins=50, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Распределение длины текстов (символы)')
axes[0].set_xlabel('Длина текста')
axes[0].set_ylabel('Количество')
axes[0].set_xlim(0, 1000)

order = ['Нейтральный', 'Позитивный', 'Негативный']
sns.boxplot(data=df, x='sentiment_name', y='text_length',
            order=order, palette='viridis', ax=axes[1], showfliers=False)
axes[1].set_title('Длина текста по классам (ящик с усами)')
axes[1].set_xlabel('Тональность')
axes[1].set_ylabel('Длина (символы)')

plt.tight_layout()
plt.show()


In [ ]:
for label, name in [(0, 'НЕЙТРАЛЬНЫЙ'), (1, 'ПОЗИТИВНЫЙ'), (2, 'НЕГАТИВНЫЙ')]:
    subset = df[df['sentiment'] == label]
    if len(subset) == 0:
        continue
    sample = subset['full_text'].sample(1, random_state=42).values[0]
    print(f'--- Пример {name} ---')
    print(sample[:500])
    print('-' * 50)
    print()

### 1.1 Предобработка

- приводим текст к нижнему регистру;
- удаляем HTML-теги, ссылки, email и упоминания;
- сохраняем русские и латинские буквы, цифры и базовую пунктуацию;
- исключаем пустые и слишком короткие строки;
- удаляем дубликаты **внутри одного split**, не перенося одинаковые тексты между train и validation;
- обрезаем экстремально длинные тексты до 10 000 символов, чтобы ограничить расход памяти на векторизации. Более 99% текстов исходного файла короче этого порога.

In [ ]:
import re

URL_RE     = re.compile(r"https?://\S+|www\.\S+")
HTML_RE    = re.compile(r"<[^>]+>")
MENTION_RE = re.compile(r"[@#]\w+")
EMAIL_RE   = re.compile(r"\S+@\S+")

# Диапазоны Unicode, покрывающие эмодзи
_EMOJI_CLASS = (
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E0-\U0001F1FF"
    "\U0000FE00-\U0000FE0F"
    "\U00002B00-\U00002BFF"
)
EMOJI_RE = re.compile("[" + _EMOJI_CLASS + "]", flags=re.UNICODE)

# ВАЖНО: та же эмодзи-класс добавлена и в токенизатор,
# иначе findall их выкидывает.
TOKEN_RE = re.compile(
    r"[а-яёa-z0-9]+(?:-[а-яёa-z0-9]+)*"
    r"|[!?]+"
    r"|[()]+"
    r"|[" + _EMOJI_CLASS + r"]",
    flags=re.UNICODE,
)


def clean_text(text: str,
               keep_emoji: bool = True,
               mask_numbers: bool = False,
               fix_yo: bool = True,
               max_len: int = 10_000) -> str:
    if not isinstance(text, str):
        return ""
    text = text[:max_len]

    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" URL ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = re.sub(r"[\n\r\t]+", " ", text)

    if keep_emoji:
        text = EMOJI_RE.sub(lambda m: f" {m.group(0)} ", text)
    else:
        text = EMOJI_RE.sub(" ", text)

    text = text.lower()
    if fix_yo:
        text = text.replace("ё", "е")

    if mask_numbers:
        text = re.sub(r"\d+([.,]\d+)?", " NUM ", text)

    text = re.sub(
        r"[^а-яa-z0-9\s!?()" + _EMOJI_CLASS + r"]",
        " ", text,
    )
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ""
    tokens = TOKEN_RE.findall(text)
    out = []
    for t in tokens:
        if len(t) > 1 and t[0] in "!?()":
            t = t[0] * 2          # «!!!!» → «!!»
        out.append(t)
    return " ".join(out)


# --- Проверка ---
demo = "Ужасное качество!!! ((( пришло за 30 дней(( размер XL не подошёл 😡😡 А цена 1 999 руб!"
print("Оригинал:", demo)
print("clean  :", clean_text(demo))
print("tok    :", tokenize(clean_text(demo)))
print()
print("Ожидание: в tok ДОЛЖНЫ быть 😡 😡 (по одному на каждый эмодзи)")

In [ ]:
# Показываем, что мы теряли при старой предобработке
sample_eda = df["full_text"].sample(min(50_000, len(df)), random_state=RANDOM_STATE).astype(str)

emoji_mask = sample_eda.str.contains(EMOJI_RE, regex=True)
paren_mask = sample_eda.str.contains(r"[()]")
bang_mask  = sample_eda.str.contains(r"[!?]")

print(f"Выборка {len(sample_eda):,} текстов:")
print(f"  • с эмодзи:        {emoji_mask.mean():.2%}")
print(f"  • с ( или ):       {paren_mask.mean():.2%}")
print(f"  • с ! или ?:       {bang_mask.mean():.2%}")

from collections import Counter
top_emoji = Counter(EMOJI_RE.findall("".join(sample_eda))).most_common(15)
print("\nТоп-15 эмодзи:")
for ch, n in top_emoji:
    print(f"  {ch}  ×{n:,}")

In [ ]:
print(f"Размер ДО очистки: {df.shape[0]:,}")

df_clean = df.copy()
df_clean = df_clean.dropna(subset=["sentiment", "split"])
df_clean["full_text"] = df_clean["full_text"].fillna("").astype(str)

t0 = time.time()
df_clean["text_clean"] = df_clean["full_text"].apply(
    lambda t: tokenize(clean_text(t, keep_emoji=True, fix_yo=True, max_len=10_000))
)
print(f"Предобработка: {time.time() - t0:.1f} с")

MIN_LEN = 10
before = len(df_clean)
df_clean = df_clean[df_clean["text_clean"].str.len() >= MIN_LEN].copy()
df_clean = df_clean.drop_duplicates(subset=["split", "text_clean"]).reset_index(drop=True)
df_clean["sentiment_name"] = df_clean["sentiment"].map(CLASS_MAP)

print(f"Удалено (пустые, короткие, дубли): {before - len(df_clean):,}")
print(f"Размер ПОСЛЕ: {len(df_clean):,}")
print("\nРазбиение:")
print(df_clean["split"].value_counts())

In [ ]:
has_emoji_orig  = df_clean[df_clean["full_text"].str.contains(EMOJI_RE, regex=True)]
has_emoji_clean = df_clean[df_clean["text_clean"].str.contains(EMOJI_RE, regex=True)]

print(f"Текстов с эмодзи в full_text:  {len(has_emoji_orig):,}")
print(f"Текстов с эмодзи в text_clean: {len(has_emoji_clean):,}")
print()

# Показываем примеры
for _, row in has_emoji_orig.head(5).iterrows():
    print(f"Ориг.: {row['full_text'][:130]}")
    print(f"Чист.: {row['text_clean'][:130]}")
    print()

In [ ]:
df_final = df_clean[["text_clean", "sentiment", "sentiment_name", "split"]].rename(
    columns={"text_clean": "text"}
)
CLEANED_PATH = Path("ru_sentiment_cleaned_v2.csv")   # <-- новое имя
df_final.to_csv(CLEANED_PATH, index=False, encoding="utf-8-sig")
print(f"Сохранено: {CLEANED_PATH.resolve()}, размер: {df_final.shape}")

In [ ]:
print('=== Распределение классов в очищенном датасете ===')
print(df_clean['sentiment'].value_counts().sort_index())
print('\nДоли (%):')
print((df_clean['sentiment'].value_counts(normalize=True).sort_index() * 100).round(2))

order = ['Нейтральный', 'Позитивный', 'Негативный']
plt.figure(figsize=(8, 5))
ax = sns.countplot(
    data=df_clean,
    x='sentiment_name',
    order=order,
    palette='viridis'
)
plt.title('Распределение классов после очистки')
plt.xlabel('Тональность')
plt.ylabel('Количество отзывов')

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

plt.show()

print(f'\nВсего после очистки: {len(df_clean)}')

In [ ]:
import pandas as pd

df = pd.read_csv("/content/merged_sentiment_dataset.csv.gz")

print("Размер датасета:", df.shape)
print("\nКолонки:", df.columns.tolist())
print("\nРаспределение классов:")
print(df["sentiment"].value_counts(dropna=False))

print("\nПервые 5 строк:")
display(df.head())

In [ ]:
print("Разделение данных:")
print(df["split"].value_counts(dropna=False))

print("\nКлассы по выборкам:")
display(pd.crosstab(df["split"], df["sentiment"]))

print("\nИсточники данных:")
print(df["source_dataset"].value_counts(dropna=False))

print("\nПропуски в текстах:")
print(df["text"].isna().sum())

print("\nПолные дубли текстов:")
print(df["text"].duplicated().sum())

In [ ]:
# Проверка пересечений и длины текстов

import pandas as pd

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()

# Нормализуем текст для проверки возможных дублей
def normalize_for_check(text):
    return " ".join(str(text).lower().split())

train_normalized = set(
    train_df["text"].map(normalize_for_check)
)

val_normalized = set(
    val_df["text"].map(normalize_for_check)
)

overlap = train_normalized & val_normalized

print("Пересечений Train / Validation:", len(overlap))

print("\nДлины текстов (в словах):")
for name, data in [
    ("Train", train_df),
    ("Validation", val_df)
]:
    lengths = data["text"].str.split().str.len()

    print(f"\n{name}:")
    print("Медиана:", lengths.median())
    print("95-й перцентиль:", lengths.quantile(0.95))
    print("Максимум:", lengths.max())

print("\nРаспределение источников по выборкам:")
display(
    pd.crosstab(
        df["source_dataset"],
        df["split"]
    )
)

In [ ]:
import re
import numpy as np

def tokenize_new(text):
    return re.findall(r"[а-яёa-z0-9]+", str(text).lower())

# Проверяем на обучающей выборке
train_lengths = train_df["text"].map(
    lambda text: len(tokenize_new(text))
)

print("Длины после токенизации:")
print(train_lengths.describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
))

print("\nДоля текстов, которые будут обрезаны:")

for max_len in [70, 128, 256, 384, 512]:
    truncated = (train_lengths > max_len).mean() * 100
    print(f"MAX_LEN={max_len}: {truncated:.2f}%")

In [ ]:
# Проверяем пустые после токенизации и экстремально длинные тексты

print("Пустых после токенизации:", (train_lengths == 0).sum())
print("Короче 3 токенов:", (train_lengths < 3).sum())
print("Длиннее 1000 токенов:", (train_lengths > 1000).sum())

print("\nПримеры самых длинных текстов:")

longest_indices = train_lengths.nlargest(3).index

for idx in longest_indices:
    text = train_df.loc[idx, "text"]

    print("\n" + "=" * 60)
    print("Длина:", train_lengths.loc[idx], "токенов")
    print("Источник:", train_df.loc[idx, "source_dataset"])
    print("Текст (первые 700 символов):")
    print(str(text)[:700])

In [ ]:
# Анализ экстремально длинных текстов

long_df = train_df.loc[train_lengths > 1000].copy()
long_df["tokens"] = train_lengths.loc[long_df.index]

print("Всего длинных текстов:", len(long_df))

print("\nРаспределение по классам:")
display(pd.crosstab(
    long_df["source_dataset"],
    long_df["sentiment"],
    margins=True
))

print("\nПримеры из разных классов:")

for sentiment_id, name in [
    (0, "Neutral"),
    (1, "Positive"),
    (2, "Negative")
]:
    examples = long_df[
        long_df["sentiment"] == sentiment_id
    ].head(2)

    print(f"\n{'=' * 60}")
    print(name)

    for _, row in examples.iterrows():
        print(
            f"\nДлина: {row['tokens']} токенов | "
            f"Источник: {row['source_dataset']}"
        )
        print(str(row["text"])[:400])

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # В исходном CSV есть отдельные экстремально длинные тексты.
    text = text[:10_000]
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[@#]\w+", " ", text)
    text = re.sub(r"[\n\r\t]+", " ", text)
    text = re.sub(r"[^а-яёa-z0-9\s.!?,\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Размер ДО очистки:", f"{df.shape[0]:,}")
df_clean = df.copy()
df_clean = df_clean.dropna(subset=["sentiment", "split"])
df_clean["full_text"] = df_clean["full_text"].fillna("").astype(str)
df_clean["text_clean"] = df_clean["full_text"].apply(clean_text)

MIN_LEN = 10
df_clean = df_clean[df_clean["text_clean"].str.len() >= MIN_LEN].copy()
# Дедупликация внутри каждой части, чтобы не менять официальное разбиение.
df_clean = df_clean.drop_duplicates(subset=["split", "text_clean"]).reset_index(drop=True)
df_clean["sentiment_name"] = df_clean["sentiment"].map(CLASS_MAP)

print("Размер ПОСЛЕ очистки:", f"{df_clean.shape[0]:,}")
print(f"Удалено: {df.shape[0] - df_clean.shape[0]:,}")
print("\nРазбиение после очистки:")
print(df_clean["split"].value_counts())
display(df_clean[["full_text", "text_clean", "sentiment_name", "split"]].sample(min(5, len(df_clean)), random_state=RANDOM_STATE))


In [ ]:
df_final = df_clean[["text_clean", "sentiment", "sentiment_name", "split"]].rename(
    columns={"text_clean": "text"}
)
CLEANED_PATH = Path("ru_sentiment_cleaned.csv")
df_final.to_csv(CLEANED_PATH, index=False, encoding="utf-8-sig")
print(f"Сохранено: {CLEANED_PATH.resolve()}, размер: {df_final.shape}")
display(df_final.head())


In [ ]:
print('=== Распределение классов в очищенном датасете ===')
print(df_clean['sentiment'].value_counts().sort_index())
print('\nДоли (%):')
print((df_clean['sentiment'].value_counts(normalize=True).sort_index() * 100).round(2))

order = ['Нейтральный', 'Позитивный', 'Негативный']
plt.figure(figsize=(8, 5))
ax = sns.countplot(
    data=df_clean,
    x='sentiment_name',
    order=order,
    palette='viridis'
)
plt.title('Распределение классов после очистки')
plt.xlabel('Тональность')
plt.ylabel('Количество отзывов')

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

plt.show()

print(f'\nВсего после очистки: {len(df_clean)}')